# 00. syncmill 소개: 무엇이고, 왜 쓰고, 어떻게 배우나

syncmill 실습 노트북 시리즈의 출발점입니다. 이 노트북은 **읽는 것이 중심**이고, 끝에서 설치가 잘 됐는지 한 번 확인합니다.
5분이면 읽을 수 있습니다.

| 이 노트북에서 | 내용 |
|---|---|
| 1 | syncmill이 푸는 문제 |
| 2 | 전체 구조 한눈에 보기 |
| 3 | 꼭 알아야 할 용어 |
| 4 | 안전 장치와 비용 |
| 5 | 시리즈 안내(01~04) |
| 6 | 자주 묻는 질문 |
| 7 | 설치 확인 |

## 1. syncmill이 푸는 문제

Claude Code, Codex CLI, Kimi Code 같은 **AI 코딩 에이전트**는 터미널에서 요청을 받아 코드를 고쳐 줍니다. 하나만 쓸 때는
충분하지만, 일을 더 많이 맡기기 시작하면 이런 문제가 생깁니다.

- 에이전트가 **내 작업 폴더를 바로** 고쳐서, 마음에 들지 않으면 되돌리기 번거롭다.
- 한 에이전트가 막히거나 사용량 한도에 걸리면 **다른 CLI를 열고 요청을 다시** 적어야 한다.
- 같은 과제를 여러 에이전트에게 시켜 **비교하거나 나눠 맡기고** 싶지만 관리가 번거롭다.
- 어제 한 에이전트가 알아낸 것을 오늘 다른 에이전트는 **모른다.**
- 할 일이 여러 개일 때 **무엇이 끝났고, 정말 검증됐는지** 기록이 없다.

**syncmill은 이 에이전트들을 지휘하는 도구입니다.** 명령 한 줄로 하나 또는 여러 에이전트에게 일을 맡기고, 결과를 검토할 수
있는 패치로 받아, 사람이 확인한 것만 내 코드에 반영합니다.

**syncmill이 하지 않는 것**도 분명히 해 둡니다.

- 직접 코드를 쓰지 않습니다. 코드는 에이전트가 씁니다.
- API 키를 따로 받지 않습니다. 각 에이전트 CLI에 로그인된 계정을 그대로 씁니다.
- 결과를 자동으로 내 코드에 합치거나 커밋하지 않습니다.

## 2. 전체 구조 한눈에 보기

```text
 여러분 ──"syncmill run <요청>"──▶ ┌──────────────── syncmill (감독자) ────────────────┐
                                   │                                                    │
                                   │  ① 저장소 스냅샷                                    │
                                   │  ② 전략에 따라 에이전트 실행 ──▶ codex / claude / kimi │
                                   │     (에이전트마다 별도 작업 폴더)                     │
                                   │  ③ 공유 메모리에서 관련 기록을 찾아 프롬프트에 붙임     │
                                   │  ④ (선택) 품질 게이트로 결과 검사                     │
                                   │  ⑤ 결과를 패치로 보관 + 성공 결과를 메모리에 기록       │
                                   └────────────────────────────────────────────────────┘
 여러분 ◀── run id ── 검토: syncmill diff <id> ── 반영: syncmill apply <id> ──▶ 내 작업 폴더
```

설계의 중심 원칙은 하나입니다. **에이전트와 메모리를 동시에 만지는 것은 syncmill(감독자)뿐입니다.** 에이전트는 "프롬프트를 받아
결과를 내는 프로그램"일 뿐이라, 서로의 존재나 메모리를 직접 알지 못합니다. 에이전트끼리 결과를 나누는 것도 syncmill이 메모리를
통해 대신 전달합니다.

## 3. 꼭 알아야 할 용어

| 용어 | 뜻 | 자세히 |
|---|---|---|
| **에이전트(agent)** | syncmill이 실행하는 AI 코딩 CLI. `claude`, `codex`, `kimi-code`(실행 파일은 `kimi`) | 01 |
| **우선순위(priority)** | 에이전트를 시도하는 기본 순서. 기본값 `claude` → `codex` → `kimi-code` | 01, 02 |
| **전략(strategy)** | 에이전트를 어떻게 함께 쓸지. `route`(기본), `pipeline`, `compete`, `council`, `decompose` | 02 |
| **작업 공간(worktree)** | 원본과 분리된 작업용 폴더. 대부분의 전략은 에이전트마다 따로 주고, `pipeline`은 한 폴더를 단계끼리 이어 씀 | 01, 02 |
| **아티팩트 / run id** | 실행 결과(패치 포함)의 보관 기록과 그 이름 | 01 |
| **`diff` / `apply`** | 결과 검토 / 원본에 반영. 반영은 언제나 명시적 | 01 |
| **품질 게이트(gate)** | 결과를 검사할 명령. 통과한 결과만 성공으로 침 | 01, 02, 04 |
| **공유 메모리(shared memory)** | 성공한 결과를 기록해 다음 작업 프롬프트에 붙여 주는 저장소 | 03 |
| **워크보드(board)** | 여러 작업의 상태·의존·증거를 기록하는 보드. 검증된 것만 `done` | 04 |

## 4. 안전 장치와 비용

**안전 장치**

| 장치 | 지켜 주는 것 |
|---|---|
| 별도 작업 폴더 + 패치 보관 | 에이전트가 만든 변경이 내 작업 폴더에 바로 섞이지 않음 |
| 명시적 `apply` | 무엇을 받아들일지 사람이 마지막에 결정 |
| `apply` 전 원본 지문 비교 | 실행 후 원본이 바뀌었으면 적용을 거부해 내 수정을 덮어쓰지 않음 |
| 품질 게이트 | 검사를 통과하지 못한 결과는 실패로 처리 |
| 보드의 `done` 규칙 | 검증(게이트 통과·사람 확인) 없이는 완료로 기록하지 않음 |

> **주의**: 작업 폴더는 보안 샌드박스가 아닙니다. 에이전트 프로그램이 무엇에 접근할 수 있는지는 각 CLI의 권한·샌드박스 설정이 맡습니다.

**비용** — syncmill 자체는 비용이 들지 않지만, 에이전트를 부를 때마다 각 CLI 계정의 사용량이 듭니다.

- `syncmill run --dry-run`으로 **최대 호출 수**를 먼저 볼 수 있습니다(에이전트를 부르지 않음).
- 전략에 따라 호출 수가 크게 다릅니다. 에이전트 2개 기준 `route` 최대 2번(보통 1번), `council` 6번, `decompose` 최대 11번(02에서 비교).
- 이 노트북 시리즈는 **가짜 에이전트**로 실행되므로 비용이 들지 않습니다. 실제 CLI를 부르는 셀은 01의 선택 셀 하나뿐이고 기본으로 꺼져 있습니다.

## 5. 시리즈 안내

| 노트북 | 배우는 것 | 시간 |
|---|---|---|
| [01. 첫 실행](01-first-run.ipynb) | `init` → `doctor` → `run` → `diff` → `apply`, 출력 읽는 법, 실패·대체 실행, 품질 게이트, `continue`, 내 프로젝트 체크리스트 | 15분 |
| [02. 전략 비교](02-strategies.ipynb) | 다섯 전략의 동작·비용·결과 선택 규칙 비교, 상황별 선택 가이드 | 10분 |
| [03. 공유 메모리](03-shared-memory.ipynb) | 기록이 저장되는 곳과 모양, 다음 작업에 붙는 방식, `serve`로 다른 도구에서 읽기 | 10분 |
| [04. 워크보드와 TUI](04-board-and-tui.ipynb) | 작업 등록·의존·팀 결정, 검증된 완료, 막힌 작업, TUI 소개 | 10분 |

**처음이라면 01부터** 순서대로 보세요. 02~04는 01을 마친 뒤라면 필요한 것부터 봐도 됩니다.

모든 노트북은 `/tmp/syncmill-notebook.*` 아래에 일회용 **샌드박스**를 만들어 그 안에서만 작업하고, 마지막 셀에서 지웁니다. 샌드박스
도우미의 동작은 [`_sandbox.py`](_sandbox.py)에서 볼 수 있습니다.

## 6. 자주 묻는 질문

**에이전트 CLI 세 개가 다 있어야 하나요?**
아니요. 하나만 설치·로그인돼 있어도 됩니다. 없는 에이전트는 건너뜁니다. 이 시리즈는 가짜 에이전트를 쓰므로 하나도 없어도 됩니다.

**내 코드가 바뀌나요?**
`syncmill apply`를 실행하기 전에는 바뀌지 않습니다. 커밋도 하지 않습니다.

**어떤 운영체제에서 되나요?**
macOS와 Linux입니다. Windows에서는 실행 명령이 거부됩니다. Python 3.12 이상과 Git이 필요합니다.

**메모리가 없거나 고장 나면요?**
작업은 그대로 진행되고, 노란 안내와 함께 공유 맥락만 빠집니다(03).

**결과가 마음에 안 들면요?**
`discard`로 버리거나, `continue`로 같은 에이전트에게 이어서 고치게 하거나, 요청을 고쳐 다시 `run`합니다(01).

**더 자세한 설명은요?**
[사용 안내서](../user-guide.ko.md)에 모든 명령과 설정이 있고, 실제 CLI로 안전하게 시험하는 방법은
[`/tmp`에서 안전하게 시험하기](../tmp-test-guide.ko.md)에 있습니다.

## 7. 설치 확인

이 노트북을 syncmill 소스 폴더에서 다음처럼 열었다면 아래 셀이 버전과 전략 목록을 보여 줍니다.

```bash
uv sync
uv run --with jupyterlab jupyter lab docs/notebooks
```

In [1]:
import subprocess
from _sandbox import syncmill_executable

for args in (["--version"], ["strategies"]):
    done = subprocess.run([syncmill_executable(), *args], capture_output=True, text=True)
    print(done.stdout.strip())
    print()

syncmill, version 0.2.0

compete    run isolated candidates and select a verified winner
council    propose, critique, and synthesize across agents
decompose  plan subtasks, execute them, then synthesize
pipeline   run sequential stages in one evolving workspace
route      try agents in priority order with fallback



**출력 읽는 법**: 첫 줄은 설치된 syncmill 버전, 그 아래는 쓸 수 있는 전략 다섯 개와 한 줄 설명입니다. 오류가 나면 위의 `uv sync`를
먼저 실행했는지 확인하세요.

준비됐으면 [01. 첫 실행](01-first-run.ipynb)으로 넘어가세요.